# LoveDA paper-faithful baselines — part 1

Backbone-pretrained mode. Each model uses the ImageNet backbone initialization declared by its pinned public config; segmentation heads are random and no trained segmentation checkpoint is loaded.

In [ ]:
from pathlib import Path

TRAIN_ROOT = Path('/kaggle/input/datasets/alienxc137/loveda-satellite-images-semantic-segmentation/Train')
VAL_ROOT = Path('/kaggle/input/datasets/alienxc137/loveda-satellite-images-semantic-segmentation/Val')
LANDCOVER = Path('/kaggle/working/LandCover')
MMSEG = Path('/kaggle/working/mmsegmentation')
DATA_ROOT = Path('/kaggle/working/loveda_mmseg')
VENV = Path('/kaggle/working/loveda_openmmlab_py311')
PYTHON = VENV / 'bin' / 'python'
OUTPUT = Path('/kaggle/working/loveda_backbone_pretrained_part1')
MODELS = ['pspnet', 'deeplabv3plus', 'danet', 'semantic_fpn', 'ocrnet']
SEED = 2025
print('Models:', MODELS)

In [ ]:
import shutil, subprocess, sys

def run(command):
    print('>', ' '.join(map(str, command)))
    subprocess.check_call([str(x) for x in command])

if not (LANDCOVER / '.git').exists():
    run(['git', 'clone', 'https://github.com/TuanHungGiang/LandCover.git', LANDCOVER])
else:
    run(['git', '-C', LANDCOVER, 'pull', '--ff-only'])
if not (MMSEG / '.git').exists():
    run(['git', 'clone', 'https://github.com/open-mmlab/mmsegmentation.git', MMSEG])
run(['git', '-C', MMSEG, 'checkout', '--detach', 'b040e147adfa027bbc071b624bedf0ae84dfc922'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = shutil.which('uv')
run([UV, 'python', 'install', '3.11'])
if not PYTHON.exists():
    run([UV, 'venv', VENV, '--python', '3.11'])
run([UV, 'pip', 'install', '--python', PYTHON, 'numpy<2', 'torch==2.1.0+cu121', 'torchvision==0.16.0+cu121', '--extra-index-url', 'https://download.pytorch.org/whl/cu121'])
run([UV, 'pip', 'install', '--python', PYTHON, 'https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/mmcv-2.1.0-cp311-cp311-manylinux1_x86_64.whl'])
run([UV, 'pip', 'install', '--python', PYTHON, 'mmengine>=0.10,<1.0', 'mmpretrain==1.2.0', 'ftfy==6.3.1', 'transformers==4.38.2'])
run([UV, 'pip', 'install', '--python', PYTHON, '-e', MMSEG])
run([PYTHON, '-c', 'import torch, mmcv, mmcv._ext, ftfy; from mmcv.ops import point_sample; import mmpretrain.models; print("OpenMMLab smoke test: OK", torch.__version__, mmcv.__version__)'])


In [ ]:
PIPELINE = LANDCOVER / 'baseline_model' / 'paper_faithful'
run([PYTHON, PIPELINE / 'prepare_loveda.py', '--train', TRAIN_ROOT, '--val', VAL_ROOT, '--output', DATA_ROOT])

In [ ]:
import torch
command = [PYTHON, PIPELINE / 'run_queue.py', '--models', *MODELS, '--mmseg-root', MMSEG, '--data-root', DATA_ROOT, '--output', OUTPUT, '--seed', str(SEED), '--gpus', str(max(1, torch.cuda.device_count()))]
run(command)

In [ ]:
import json, shutil
from IPython.display import display
import pandas as pd

summary_path = OUTPUT / 'summary.json'
if summary_path.exists():
    rows = json.loads(summary_path.read_text())
    display(pd.json_normalize(rows))
    archive = shutil.make_archive(str(OUTPUT), 'zip', root_dir=OUTPUT)
    print('Archive:', archive)
else:
    print('No summary yet. Inspect per-model status.json and logs in', OUTPUT)